# Load packages

In [1]:
import scanpy as sc
import pandas as pd

# Load Data

In [8]:
test = pd.read_csv('/Volumes/group/h345/afarkkilab/Data/17_Auria_TMA/for_Silja/data_06052024.csv', index_col=0)

In [20]:
test_2 = pd.read_csv('/Volumes/group/h345/afarkkilab/Data/17_Auria_TMA/for_Silja/data_RCNs_Auria.csv', index_col=0)

In [23]:
test.columns

Index(['H3.1', 'pRPA32', 'S9.6', 'Ki67_1', 'TNFR1', 'HER3', 'panCK', 'SMA',
       'CD45', 'Ki67_2', 'CD4', 'FOXP3', 'CD8a', 'CD11b', 'CD68', 'CD206',
       'PCNA', 'ER', 'HER2', 'EGFR', 'PR', 'H2ax', 'Vimentin', 'H3K27me3',
       'E.Cadherin', 'GCLC', 'TXNRD1', 'NQO1', 'pERK', 'TXN', 'pSTAT3',
       'pS6_235', 'pRB', 'GLUT1', 'X_centroid', 'Y_centroid', 'imageid',
       'Area', 'Eccentricity', 'celltype', 'antioxidant_NQOI',
       'antioxidant_GCLC', 'antioxidant_TXNRD1', 'antioxidant_GLUT1',
       'cores.x', 'patient_id_AB19_1654', 'Annotation', 'Stage',
       'core_imageid', 'spatial_kmeans', 'marker_Ki67', 'marker_PCNA',
       'classified_NQO1_Vim', 'classified_GLUT1_Vim'],
      dtype='object')

In [2]:
data_raw = pd.read_csv('/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/Data_AllMarkers_filtered.csv')

/var/folders/rq/68pl8_s15_lc84mft22mqf012ws2td/T/ipykernel_20851/2776709910.py:1: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  data_raw = pd.read_csv('/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/Data_AllMarkers_filtered.csv')


In [4]:
data_raw.columns

Index(['CellID', 'S9.6', 'HER3', 'panCK', 'SMA', 'CD45', 'Ki67_2', 'CD4',
       'FOXP3', 'CD8a', 'CD11b', 'CD68', 'CD206', 'H2ax', 'Vimentin',
       'H3K27me3', 'E.Cadherin', 'GCLC', 'TXNRD1', 'NQO1', 'pERK', 'TXN',
       'pSTAT3', 'pS6_235', 'pRB', 'GLUT1', 'X_centroid', 'Y_centroid',
       'imageid', 'core_imageid', 'cores.x', 'celltype', 'Annotation', 'Stage',
       'patient_id_AB19_1654', 'Area', 'Eccentricity', 'celltype.1',
       'antioxidant_NQOI', 'antioxidant_GCLC', 'antioxidant_TXNRD1',
       'antioxidant_GLUT1', 'NQO1_GCLC_VIM', 'celltype_2'],
      dtype='object')

# Add annotations

In [3]:
data_raw['celltype.1'].unique()

array(['Other', 'Tumor', 'CD68.Macrophages', 'Immune', 'FOXP3.CD4.Tregs',
       'CD4.T.cells', 'Myofibroblasts', 'CD206.Macrophages',
       'CD8.T.cells'], dtype=object)

In [4]:
# Create a new column celltype_1 with the values from celltype.1
data_raw['celltype_1'] = data_raw['celltype.1']

# Reclassify "Other" cells based on Vimentin expression
mask_other = data_raw['celltype_1'] == 'Other'
data_raw.loc[mask_other & (data_raw['Vimentin'] >= 0.5), 'celltype_1'] = 'VIM+.Stroma'
data_raw.loc[mask_other & (data_raw['Vimentin'] < 0.5), 'celltype_1'] = 'Other.Stroma'

# Change 'Immune' to 'Other_Immune'
# Change 'FOXP3.CD4.Tregs' to 'Treg'
data_raw.loc[data_raw['celltype_1'] == 'Immune', 'celltype_1'] = 'Other.Immune'
data_raw.loc[data_raw['celltype_1'] == 'FOXP3.CD4.Tregs', 'celltype_1'] = 'Treg'
data_raw.loc[data_raw['celltype_1'] == 'Myofibroblasts', 'celltype_1'] = 'SMA+.Stroma'

# Drop the original celltype.1 column
data_raw = data_raw.drop('celltype.1', axis=1)

In [5]:
data_raw['celltype_1'].unique()

array(['Other.Stroma', 'Tumor', 'CD68.Macrophages', 'Other.Immune',
       'Treg', 'CD4.T.cells', 'SMA+.Stroma', 'CD206.Macrophages',
       'VIM+.Stroma', 'CD8.T.cells'], dtype=object)

In [6]:
# Create a new column Tumor_state with the values from celltype_1
data_raw['Tumor_state'] = data_raw['celltype_1']

# Reclassify "Tumor" cells based on Vimentin expression
mask_other = data_raw['Tumor_state'] == 'Tumor'
data_raw.loc[mask_other & (data_raw['Vimentin'] >= 0.5), 'Tumor_state'] = 'VIM+'
data_raw.loc[mask_other & (data_raw['Vimentin'] < 0.5), 'Tumor_state'] = 'Tumor'


In [13]:
data_raw['GCLC_VIM'] = data_raw['celltype_1']

mask_other = data_raw['GCLC_VIM'] == 'Tumor'
data_raw.loc[mask_other & (data_raw['GCLC'] >= 0.5) & (data_raw['Vimentin'] >= 0.5), 'GCLC_VIM'] = 'Tumor.GCLC+VIM+'
data_raw.loc[mask_other & (data_raw['GCLC'] >= 0.5) & (data_raw['Vimentin'] < 0.5), 'GCLC_VIM'] = 'Tumor.GCLC+VIM-'
data_raw.loc[mask_other & (data_raw['GCLC'] < 0.5) & (data_raw['Vimentin'] >= 0.5), 'GCLC_VIM'] = 'Tumor.GCLC-VIM+'
data_raw.loc[mask_other & (data_raw['GCLC'] < 0.5) & (data_raw['Vimentin'] < 0.5), 'GCLC_VIM'] = 'Tumor.GCLC-VIM-'

In [14]:
data_raw['GCLC_VIM'].unique()

array(['Other.Stroma', 'Tumor.GCLC-VIM-', 'CD68.Macrophages',
       'Other.Immune', 'Treg', 'Tumor.GCLC+VIM-', 'CD4.T.cells',
       'SMA+.Stroma', 'Tumor.GCLC-VIM+', 'CD206.Macrophages',
       'VIM+.Stroma', 'CD8.T.cells', 'Tumor.GCLC+VIM+'], dtype=object)

In [15]:
Tumor = ['Tumor']
Stroma = ['SMA+.Stroma', 'VIM+.Stroma', 'Other.Stroma']
Immune = ['CD4.T.cells', 'CD8.T.cells', 'Treg', 'CD68.Macrophages', 'CD206.Macrophages', 'Other.Immune']

def determin_cell_category(celltype):
    if celltype in Tumor:
        return 'Tumor'
    if celltype in Stroma:
        return 'Stroma'
    if celltype in Immune:
        return 'Immune'
    return 'other'

data_raw['CellCategory'] = data_raw['celltype_1'].apply(determin_cell_category)
data_raw['CellCategory'].value_counts()

CellCategory
Tumor     1773160
Immune    1123431
Stroma     785181
Name: count, dtype: int64

In [17]:
data_raw.to_csv('/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/Data_AllMarkers_filtered_2.csv', index=False)

# Data info

In [18]:
# chage the type of 'Stage' to string
data_raw['Stage'] = data_raw['Stage'].astype(str)

patient_stage = data_raw.groupby(['patient_id_AB19_1654', 'Stage']).size().reset_index(name='counts')
patient_stage['Stage'].value_counts()

Stage
2          91
1          49
unknown    33
3          11
4           2
Name: count, dtype: int64

In [19]:
data_raw['patient_id_AB19_1654'].nunique()

186

In [20]:
core_annotation = data_raw.groupby(['core_imageid', 'Annotation']).size().reset_index(name='counts')
core_annotation['Annotation'].value_counts()

Annotation
Tumor center       192
Invasive border     66
Inflammation        61
Lymph node met      26
Name: count, dtype: int64